In [1]:
"""
data_generator.py
-----------------
Generates a realistic, diverse review dataset (600 samples, 200 per class).
Avoids template repetition through combinatorial variation across:
  - sentence structures
  - product domains (electronics, food, clothing, service, software)
  - intensity modifiers
  - contextual phrases

This ensures the model is learning generalizable sentiment geometry,
not memorizing surface-level patterns from tiny repeated templates.

Run:
    python data_generator.py
Output:
    data/reviews.csv  (600 rows, columns: review, label)
"""

import csv
import random
import os
from itertools import product

random.seed(42)

# ─────────────────────────────────────────────────────────────────────────────
# VOCABULARY BANKS
# ─────────────────────────────────────────────────────────────────────────────

DOMAINS = {
    'electronics': [
        "laptop", "phone", "headphones", "camera", "tablet", "keyboard",
        "monitor", "speaker", "smartwatch", "charger", "router", "earbuds"
    ],
    'food': [
        "restaurant", "meal", "food", "dish", "coffee", "pizza", "burger",
        "dessert", "sushi", "pasta", "salad", "sandwich"
    ],
    'clothing': [
        "jacket", "shirt", "shoes", "dress", "jeans", "coat", "sneakers",
        "hoodie", "suit", "bag", "hat", "boots"
    ],
    'service': [
        "customer service", "support team", "delivery", "shipping",
        "staff", "experience", "service", "team", "response", "help desk"
    ],
    'software': [
        "app", "software", "tool", "platform", "interface", "subscription",
        "update", "feature", "plugin", "dashboard", "API", "product"
    ]
}

# Flatten all items
ALL_ITEMS = [item for items in DOMAINS.values() for item in items]

POSITIVE_TEMPLATES = [
    "Absolutely love this {item}. {detail}. Would buy again.",
    "The {item} exceeded every expectation I had. {detail}.",
    "This is hands down the best {item} I have ever owned. {detail}.",
    "Incredibly impressed with this {item}. {detail}. Five stars easily.",
    "Outstanding {item}. {detail}. Highly recommend to everyone.",
    "My {item} arrived quickly and works flawlessly. {detail}.",
    "Cannot believe how good this {item} is for the price. {detail}.",
    "Genuinely amazed by the quality of this {item}. {detail}.",
    "So happy with my purchase. The {item} is {adj_pos} and {adj_pos2}.",
    "Best decision I made this year was buying this {item}. {detail}.",
    "Wow, this {item} blew me away. {detail}. Will definitely recommend.",
    "Solid build quality and {adj_pos} performance. This {item} delivers.",
    "I was skeptical at first but this {item} truly delivered. {detail}.",
    "The {item} is even better than advertised. {detail}.",
    "Top tier {item}. {detail}. You will not be disappointed.",
    "Worth every penny. This {item} is {adj_pos} and {adj_pos2}.",
    "I bought this {item} as a gift and the recipient loved it. {detail}.",
    "Exceptional {item}. {detail}. Ordering another one for sure.",
    "Just got my {item} and I am beyond satisfied. {detail}.",
    "Premium feel, premium results. This {item} is {adj_pos} in every way.",
]

NEGATIVE_TEMPLATES = [
    "Terrible {item}. {detail}. Complete waste of money.",
    "This {item} broke within {timeframe}. {detail}. Never buying again.",
    "Worst {item} I have ever purchased. {detail}. Avoid at all costs.",
    "Extremely disappointed with this {item}. {detail}.",
    "Do not buy this {item}. {detail}. Absolute garbage.",
    "The {item} stopped working after {timeframe}. {detail}. Unacceptable.",
    "False advertising. This {item} is nothing like described. {detail}.",
    "Returned this {item} immediately. {detail}. Total scam.",
    "I cannot believe this {item} is sold legally. {detail}. Disgraceful.",
    "The {item} is {adj_neg} and {adj_neg2}. {detail}. Very upset.",
    "Horrible experience. The {item} failed to do what it promised. {detail}.",
    "Cheap build quality and {adj_neg} results. This {item} is a disaster.",
    "Paid premium price for a {adj_neg} {item}. {detail}. Outrageous.",
    "This {item} damaged my property. {detail}. Completely unacceptable.",
    "Ordered the {item} for a special occasion. It arrived broken. {detail}.",
    "Customer service was unhelpful when my {item} malfunctioned. {detail}.",
    "The {item} is {adj_neg} and falls apart immediately. {detail}.",
    "Save your money. This {item} is {adj_neg} junk. {detail}.",
    "Regret buying this {item} every single day. {detail}.",
    "Zero stars if I could. The {item} is {adj_neg} and {adj_neg2}. {detail}.",
]

NEUTRAL_TEMPLATES = [
    "The {item} is okay. {detail}. Nothing special but gets the job done.",
    "Average {item}. {detail}. Does what it is supposed to, nothing more.",
    "Not bad, not great. The {item} is a middle-of-the-road product. {detail}.",
    "The {item} meets basic expectations. {detail}. Would not rave about it.",
    "Decent enough {item} for the price. {detail}. Fine if you are on a budget.",
    "The {item} works as described. {detail}. Nothing to write home about.",
    "I have mixed feelings about this {item}. {detail}. Some good, some bad.",
    "Mediocre {item}. {detail}. There are probably better options out there.",
    "The {item} is functional but unremarkable. {detail}.",
    "Acceptable quality for the price. The {item} is {adj_neu}. {detail}.",
    "The {item} does the job. {detail}. Would consider other options next time.",
    "Received my {item} on time. {detail}. Functionality is about average.",
    "Used the {item} for {timeframe} now. {detail}. It is holding up okay.",
    "Nothing wrong with the {item} per se, but {detail}. Just underwhelming.",
    "The {item} is passable. {detail}. Not something I would gift someone.",
    "For casual use the {item} is fine. {detail}. Power users may want more.",
    "The {item} has its pros and cons. {detail}. Ends up being about average.",
    "Somewhat satisfied with the {item}. {detail}. Room for improvement.",
    "The {item} works but feels a bit {adj_neu}. {detail}.",
    "Neutral opinion on this {item}. {detail}. Not excited but not upset either.",
]

# Detail phrases by sentiment
POSITIVE_DETAILS = [
    "Build quality is phenomenal",
    "Performance is lightning fast",
    "The design is sleek and elegant",
    "Setup was incredibly easy",
    "Battery life is outstanding",
    "Delivery was faster than expected",
    "Packaging was superb",
    "Customer support was extremely responsive",
    "Works seamlessly right out of the box",
    "Exceeded the listed specifications",
    "Every feature works as advertised",
    "The material feels premium and durable",
    "Colors are vibrant and accurate",
    "Sound quality is crystal clear",
    "Performance improvement was immediately noticeable",
    "Fits perfectly and feels comfortable",
    "The interface is intuitive and smooth",
    "Highly polished experience from start to finish",
    "No issues whatsoever after extended use",
    "Absolutely worth the premium price tag",
]

NEGATIVE_DETAILS = [
    "Build quality is shockingly poor",
    "Performance is unbearably slow",
    "Design looks nothing like the photos",
    "Setup took hours and still did not work",
    "Battery drains completely in under an hour",
    "Delivery was two weeks late",
    "Arrived in damaged packaging",
    "Customer support was completely useless",
    "Required five reboots just to get started",
    "Specs listed are completely fabricated",
    "Half the advertised features do not function",
    "Materials feel like cheap plastic",
    "Colors are dull and inaccurate",
    "Sound is distorted and full of static",
    "Performance degraded rapidly within days",
    "Fits terribly and is extremely uncomfortable",
    "Interface crashes constantly",
    "Riddled with bugs and glitches throughout",
    "Failed completely after less than a week",
    "Absolutely not worth any amount of money",
]

NEUTRAL_DETAILS = [
    "Build quality is about average",
    "Performance is acceptable for the price point",
    "Design is generic but inoffensive",
    "Setup took a fair amount of time",
    "Battery life is neither great nor terrible",
    "Delivery arrived within the estimated window",
    "Packaging was standard",
    "Customer support responded after a few days",
    "Took some time to get working properly",
    "Meets the stated specifications, nothing more",
    "Features work but feel unpolished",
    "Materials feel standard for the price",
    "Colors are acceptable but a bit washed out",
    "Sound is okay for background listening",
    "Performance is consistent but not impressive",
    "Sizing runs slightly small but wearable",
    "Interface is functional but dated",
    "Experience is fine for basic use cases",
    "Performance has been consistent so far",
    "Represents fair value but not great value",
]

POSITIVE_ADJ = [
    "excellent", "fantastic", "premium", "outstanding", "superb",
    "remarkable", "impressive", "exceptional", "brilliant", "flawless"
]
POSITIVE_ADJ2 = [
    "reliable", "durable", "well-crafted", "intuitive", "responsive",
    "efficient", "polished", "powerful", "versatile", "sleek"
]
NEGATIVE_ADJ = [
    "terrible", "awful", "dreadful", "appalling", "abysmal",
    "pathetic", "shoddy", "defective", "worthless", "broken"
]
NEGATIVE_ADJ2 = [
    "unreliable", "flimsy", "poorly-made", "confusing", "unresponsive",
    "inefficient", "unfinished", "weak", "useless", "overpriced"
]
NEUTRAL_ADJ = [
    "average", "ordinary", "standard", "basic", "acceptable",
    "mediocre", "middling", "so-so", "unremarkable", "passable"
]
TIMEFRAMES = [
    "one day", "two days", "a week", "three days", "four days",
    "a few uses", "the first use", "one month", "two weeks"
]


def generate_review(template: str, sentiment: str) -> str:
    item = random.choice(ALL_ITEMS)
    if sentiment == 'positive':
        detail  = random.choice(POSITIVE_DETAILS)
        adj_pos = random.choice(POSITIVE_ADJ)
        adj_pos2 = random.choice(POSITIVE_ADJ2)
        filled = template.format(
            item=item, detail=detail, adj_pos=adj_pos,
            adj_pos2=adj_pos2, timeframe=random.choice(TIMEFRAMES)
        )
    elif sentiment == 'negative':
        detail   = random.choice(NEGATIVE_DETAILS)
        adj_neg  = random.choice(NEGATIVE_ADJ)
        adj_neg2 = random.choice(NEGATIVE_ADJ2)
        filled = template.format(
            item=item, detail=detail, adj_neg=adj_neg,
            adj_neg2=adj_neg2, timeframe=random.choice(TIMEFRAMES)
        )
    else:
        detail  = random.choice(NEUTRAL_DETAILS)
        adj_neu = random.choice(NEUTRAL_ADJ)
        filled = template.format(
            item=item, detail=detail, adj_neu=adj_neu,
            timeframe=random.choice(TIMEFRAMES)
        )
    return filled


def generate_dataset(n_per_class: int = 200, output_path: str = "data/reviews.csv"):
    """
    Generate n_per_class samples for each of: positive, negative, neutral.
    Uses cycling + randomization to maximize lexical diversity.
    """
    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    rows = []

    for sentiment, templates in [
        ('positive', POSITIVE_TEMPLATES),
        ('negative', NEGATIVE_TEMPLATES),
        ('neutral',  NEUTRAL_TEMPLATES),
    ]:
        generated = set()
        attempts  = 0
        while len(generated) < n_per_class and attempts < n_per_class * 10:
            tmpl   = random.choice(templates)
            review = generate_review(tmpl, sentiment)
            if review not in generated:
                generated.add(review)
                rows.append({'review': review, 'label': sentiment})
            attempts += 1

    random.shuffle(rows)

    with open(output_path, 'w', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=['review', 'label'])
        writer.writeheader()
        writer.writerows(rows)

    print(f"✅ Generated {len(rows)} reviews → {output_path}")
    counts = {}
    for r in rows:
        counts[r['label']] = counts.get(r['label'], 0) + 1
    print(f"   Distribution: {counts}")


if __name__ == "__main__":
    generate_dataset(n_per_class=200, output_path="data/reviews.csv")

✅ Generated 600 reviews → data/reviews.csv
   Distribution: {'positive': 200, 'negative': 200, 'neutral': 200}


In [2]:
"""
preprocess.py
-------------
Minimal text preprocessing for SBERT-based pipeline.

IMPORTANT DESIGN DECISION:
    SBERT (Sentence-BERT) was trained on raw, natural language sentences.
    Aggressive preprocessing — stemming, stopword removal, lowercasing —
    DESTROYS the semantic signal SBERT was trained to encode.

    For example:
        "not amazing"  →  after stemming/stopword removal  →  "amaz"
        SBERT correctly embeds "not amazing" as semantically negative.
        The preprocessed version loses the negation entirely.

    Therefore: preprocessing is limited to whitespace normalization
    and basic Unicode cleaning only. SBERT handles the rest.
"""

import re
import pandas as pd


def clean_text(text: str) -> str:
    """
    Minimal cleaning:
      - Strip leading/trailing whitespace
      - Collapse internal whitespace runs
      - Remove non-printable control characters
      - Normalize smart quotes / apostrophes
    """
    text = text.strip()
    text = re.sub(r'[\x00-\x1F\x7F]', ' ', text)          # control chars
    text = re.sub(r'[\u2018\u2019]', "'", text)            # smart quotes
    text = re.sub(r'[\u201C\u201D]', '"', text)            # smart double quotes
    text = re.sub(r'\s+', ' ', text)                       # collapse whitespace
    return text


def load_and_preprocess(filepath: str) -> pd.DataFrame:
    """
    Load CSV and apply minimal cleaning.
    Expected columns: review, label
    Returns DataFrame with added 'clean_review' column.
    """
    df = pd.read_csv(filepath)

    required_cols = {'review', 'label'}
    if not required_cols.issubset(df.columns):
        raise ValueError(f"CSV must contain columns: {required_cols}. "
                         f"Found: {set(df.columns)}")

    df = df.dropna(subset=['review', 'label']).reset_index(drop=True)
    df['clean_review'] = df['review'].apply(clean_text)

    label_dist = df['label'].value_counts().to_dict()
    print(f"✅ Loaded {len(df)} reviews | Distribution: {label_dist}")
    return df


if __name__ == "__main__":
    df = load_and_preprocess("data/reviews.csv")
    print(df[['review', 'clean_review']].head(5))

✅ Loaded 600 reviews | Distribution: {'positive': 200, 'negative': 200, 'neutral': 200}
                                              review  \
0  So happy with my purchase. The headphones is r...   
1  Genuinely amazed by the quality of this phone....   
2  Do not buy this shipping. Design looks nothing...   
3  The jeans is even better than advertised. Abso...   
4  Neutral opinion on this team. Performance is c...   

                                        clean_review  
0  So happy with my purchase. The headphones is r...  
1  Genuinely amazed by the quality of this phone....  
2  Do not buy this shipping. Design looks nothing...  
3  The jeans is even better than advertised. Abso...  
4  Neutral opinion on this team. Performance is c...  


In [8]:
"""
vectorizer.py
-------------
Stage 1: SBERT Sentence Embeddings
Stage 2: UMAP Manifold Reduction (cosine metric)

WHY SBERT OVER TF-IDF:
    TF-IDF encodes vocabulary overlap. Two reviews:
        "Absolutely terrible, complete garbage."
        "Utterly horrible, total junk."
    → TF-IDF cosine similarity ≈ 0.0   (zero word overlap)
    → SBERT cosine similarity ≈ 0.89   (same semantic meaning)

    Sentiment clustering requires semantic proximity, not lexical overlap.
    SBERT maps semantically equivalent texts to nearby points on the
    unit hypersphere — the correct geometry for spherical K-Means.

WHY UMAP BEFORE CLUSTERING:
    SBERT produces 384-dimensional embeddings. In 384D, the Curse of
    Dimensionality causes all pairwise distances to converge toward
    the same value — clusters become invisible. UMAP:
      1. Preserves local cosine similarity structure (metric='cosine')
      2. Compresses to a dense, cluster-friendly manifold
      3. Dramatically improves silhouette geometry without losing topology
"""

import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.preprocessing import normalize
import umap


# ─────────────────────────────────────────────────────────────────────────────
# SBERT EMBEDDING
# ─────────────────────────────────────────────────────────────────────────────

def get_sbert_embeddings(
    texts: list[str],
    model_name: str = 'siebert/sentiment-roberta-large-english',
    batch_size: int = 64,
    normalize_output: bool = True
) -> np.ndarray:
    """
    Encode sentences using SBERT.

    Model choice — 'all-MiniLM-L6-v2':
      - 384D output (compact but highly expressive)
      - Trained on 1B+ sentence pairs via contrastive learning
      - Optimized for semantic textual similarity
      - ~22M parameters — fast inference even on CPU
      - State-of-the-art performance on STS benchmarks

    Args:
        texts           : list of raw (minimally cleaned) reviews
        model_name      : HuggingFace model identifier
        batch_size      : sentences encoded per forward pass
        normalize_output: L2-normalize → projects onto unit hypersphere
                          Required for cosine similarity = dot product
                          Required for valid spherical K-Means operation

    Returns:
        np.ndarray of shape (N, 384), float32, L2-normalized if requested
    """
    print(f"── Loading SBERT model: {model_name} ──")
    model = SentenceTransformer(model_name)

    print(f"── Encoding {len(texts)} sentences (batch_size={batch_size}) ──")
    embeddings = model.encode(
        texts,
        batch_size=batch_size,
        show_progress_bar=True,
        convert_to_numpy=True,
        normalize_embeddings=normalize_output   # SBERT-level L2 normalization
    )
    print(f"✅ SBERT embeddings: {embeddings.shape}  |  dtype: {embeddings.dtype}")
    return embeddings


# ─────────────────────────────────────────────────────────────────────────────
# UMAP REDUCTION
# ─────────────────────────────────────────────────────────────────────────────

def get_umap_embedding(
    X: np.ndarray,
    n_components: int = 15,
    n_neighbors: int = 15,
    min_dist: float = 0.05,
    spread: float = 1.0,
    random_state: int = 42
) -> np.ndarray:
    """
    UMAP dimensionality reduction preserving cosine similarity manifold.

    Parameter guidance:
        n_components: Target dimensions for clustering input.
                      15–30 is ideal — low enough to escape the curse of
                      dimensionality, high enough to preserve cluster structure.
                      (2D is ONLY for visualization — never cluster in 2D)

        n_neighbors:  Controls local vs global structure preservation.
                      Rule of thumb: sqrt(N) for small datasets, 15–50 for large.
                      Too small → noisy, disconnected manifold.
                      Too large → over-smoothed, clusters merge.

        min_dist:     Minimum distance between points in embedding space.
                      Low (0.0–0.1) → tight, compact clusters (good for silhouette).
                      High (0.5–1.0) → spread out, better for visualization.

        metric:       CRITICAL — must be 'cosine' to preserve the geometry
                      of SBERT's unit-hypersphere output.

    Returns:
        np.ndarray of shape (N, n_components), L2-normalized (ready for
        spherical K-Means — all vectors on the unit hypersphere)
    """
    print(f"── UMAP reduction: {X.shape[1]}D → {n_components}D  "
      f"(n_neighbors={n_neighbors}, min_dist={min_dist}) ──")

    reducer = umap.UMAP(
        n_components=n_components,
        metric='cosine',            # Must match SBERT's similarity metric
        n_neighbors=n_neighbors,
        min_dist=min_dist,
        spread=spread,
        random_state=random_state,
        low_memory=False,           # Full speed — use low_memory=True on <16GB RAM
        verbose=False
    )
    X_umap = reducer.fit_transform(X)

    # Re-normalize after UMAP — UMAP does not preserve unit norm
    X_umap_norm = normalize(X_umap, norm='l2')
    print(f"✅ UMAP embedding: {X_umap_norm.shape}  |  L2-normalized ✓")
    return X_umap_norm


def get_umap_2d(
    X: np.ndarray,
    n_neighbors: int = 15,
    min_dist: float = 0.1,
    random_state: int = 42
) -> np.ndarray:
    """
    Separate 2D UMAP projection FOR VISUALIZATION ONLY.
    Never use this for clustering — 2D loses too much structure.
    """
    reducer = umap.UMAP(
        n_components=2,
        metric='cosine',
        n_neighbors=n_neighbors,
        min_dist=min_dist,
        random_state=random_state,
        verbose=False
    )
    return reducer.fit_transform(X)


if __name__ == "__main__":
    from preprocess import load_and_preprocess
    df = load_and_preprocess("data/reviews.csv")
    X  = get_sbert_embeddings(df['clean_review'].tolist())
    X_umap = get_umap_embedding(X, n_components=15, n_neighbors=15)
    print(f"\nFinal shape for clustering: {X_umap.shape}")

✅ Loaded 600 reviews | Distribution: {'positive': 200, 'negative': 200, 'neutral': 200}
── Loading SBERT model: siebert/sentiment-roberta-large-english ──


No sentence-transformers model found with name siebert/sentiment-roberta-large-english. Creating a new one with mean pooling.


Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: siebert/sentiment-roberta-large-english
Key                             | Status     | 
--------------------------------+------------+-
classifier.out_proj.weight      | UNEXPECTED | 
classifier.dense.weight         | UNEXPECTED | 
classifier.out_proj.bias        | UNEXPECTED | 
classifier.dense.bias           | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
pooler.dense.bias               | MISSING    | 
pooler.dense.weight             | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


── Encoding 600 sentences (batch_size=64) ──


Batches:   0%|          | 0/10 [00:00<?, ?it/s]

✅ SBERT embeddings: (600, 1024)  |  dtype: float32
── UMAP reduction: 1024D → 15D  (n_neighbors=15, min_dist=0.05) ──


C:\Users\Adwaith\AppData\Roaming\Python\Python313\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


✅ UMAP embedding: (600, 15)  |  L2-normalized ✓

Final shape for clustering: (600, 15)


In [9]:
"""
geometric_kmeans.py
-------------------
Spherical K-Means++ with Curvature-Aware Centroid Updates.

MATHEMATICAL FOUNDATION
━━━━━━━━━━━━━━━━━━━━━━
After L2 normalization, all data vectors lie on S^(d-1) — the surface of
the d-dimensional unit hypersphere — which is a Riemannian manifold with
constant positive curvature.

PROBLEM WITH STANDARD K-MEANS ON THIS MANIFOLD:
    Centroid ← arithmetic mean of assigned vectors
    The arithmetic mean of unit vectors does NOT lie on the unit sphere.
    Projecting it back (renormalization) is an approximation that ignores
    the manifold's intrinsic geometry.

OUR APPROACH — Three geometric corrections:

1. INITIALIZATION: Geodesic K-Means++
   Standard K-Means++ seeds centroids with probability ∝ d_euclidean².
   We replace Euclidean distance with geodesic (arc-length) distance:
       d_geo(u, v) = arccos(u · v)   ∈ [0, π]
   This correctly spreads seeds across the sphere surface.

2. ASSIGNMENT: Cosine Similarity (exact for unit vectors)
   argmin_c d_geo(x, c) ≡ argmax_c (x · c)   when ‖x‖=‖c‖=1
   So assignment is just a matrix multiplication — fast and exact.

3. UPDATE: Curvature-Aware Fréchet Mean
   The Fréchet mean on S^(d-1) is the point that minimizes the sum of
   squared geodesic distances to all assigned vectors. It is approximated
   by a geodesic Gaussian-weighted mean:

       weight_i = exp( -d_geo(x_i, c)² / 2σ² )
       new_c    = normalize( Σ weight_i · x_i )

   Points far along the geodesic from the current centroid are
   DOWN-WEIGHTED. This:
     - Reduces pull from cluster-boundary/outlier points
     - Respects the sphere's curvature (contribution decays with arc length)
     - Converges to true Fréchet mean as σ → 0
     - Degenerates to standard spherical centroid as σ → ∞

   σ (sigma) is a hyperparameter controlling kernel width. It is tuned
   automatically via silhouette-score-guided sweep in this file.
"""

import numpy as np
from sklearn.metrics import silhouette_score


# ─────────────────────────────────────────────────────────────────────────────
# GEOMETRIC PRIMITIVES
# ─────────────────────────────────────────────────────────────────────────────

def geodesic_distances_to_centroid(X: np.ndarray, centroid: np.ndarray) -> np.ndarray:
    """
    Compute arc-length distances from all rows of X to a single centroid.
    Both X and centroid must be L2-normalized (on unit hypersphere).

    d_geo(x, c) = arccos( clip(x · c, -1, 1) )

    The clip is essential — floating point errors can push dot products
    slightly outside [-1, 1], causing arccos to return NaN.

    Returns: np.ndarray of shape (N,) in [0, π]
    """
    dot_products = np.clip(X @ centroid, -1.0, 1.0)
    return np.arccos(dot_products)


def curvature_aware_centroid(
    vectors: np.ndarray,
    current_centroid: np.ndarray,
    sigma: float
) -> np.ndarray:
    """
    Geodesic Gaussian-weighted Fréchet Mean.

    Args:
        vectors         : (M, d) — L2-normalized vectors assigned to this cluster
        current_centroid: (d,)   — current centroid (L2-normalized)
        sigma           : kernel width in radians (tuned externally)

    Returns: (d,) — updated centroid, L2-normalized (back on sphere)
    """
    geodesic_dists = geodesic_distances_to_centroid(vectors, current_centroid)
    weights = np.exp(-(geodesic_dists ** 2) / (2.0 * sigma ** 2))  # (M,)

    # Weighted mean in embedding space
    weighted_mean = (weights[:, None] * vectors).sum(axis=0)
    norm = np.linalg.norm(weighted_mean)

    if norm < 1e-12:
        # Degenerate: all weights collapsed — fall back to uniform Fréchet mean
        fallback = vectors.mean(axis=0)
        norm_fb = np.linalg.norm(fallback)
        if norm_fb < 1e-12:
            return current_centroid.copy()  # Keep old centroid unchanged
        return fallback / norm_fb

    return weighted_mean / norm   # Project back onto S^(d-1)


# ─────────────────────────────────────────────────────────────────────────────
# K-MEANS++ INITIALIZATION (GEODESIC)
# ─────────────────────────────────────────────────────────────────────────────

def geodesic_kmeans_pp_init(
    X: np.ndarray,
    k: int,
    rng: np.random.Generator
) -> np.ndarray:
    """
    K-Means++ seeding adapted for the unit hypersphere.

    Seeding strategy:
        c_1 ~ Uniform(X)
        c_i ~ Categorical( d_geo(x, nearest_c)² / Z )  for i = 2..k

    Using geodesic² (instead of Euclidean²) correctly captures angular
    separation between candidate centroids on the sphere surface.

    Returns: (k, d) array of initial centroids, all L2-normalized
    """
    n = len(X)
    first_idx = rng.integers(n)
    centroids = [X[first_idx].copy()]

    for _ in range(k - 1):
        # (N, current_k) geodesic distance matrix to all current centroids
        dist_matrix = np.column_stack([
            geodesic_distances_to_centroid(X, c) for c in centroids
        ])
        # Distance to nearest centroid for each point
        min_dists = dist_matrix.min(axis=1)           # (N,)
        probs     = min_dists ** 2
        probs    /= probs.sum()
        next_idx  = rng.choice(n, p=probs)
        centroids.append(X[next_idx].copy())

    return np.array(centroids)   # (k, d)


# ─────────────────────────────────────────────────────────────────────────────
# SPHERICAL K-MEANS++ — MAIN ALGORITHM
# ─────────────────────────────────────────────────────────────────────────────

def run_spherical_kmeans(
    X: np.ndarray,
    k: int = 3,
    sigma: float = 0.5,
    max_iter: int = 500,
    n_init: int = 20,
    tol: float = 1e-6,
    random_state: int = 42
) -> tuple[np.ndarray, np.ndarray, float]:
    """
    Spherical K-Means++ with Curvature-Aware Centroid Updates.

    Args:
        X            : (N, d) L2-normalized feature matrix (on unit sphere)
        k            : number of clusters
        sigma        : geodesic kernel width for curvature-aware centroid
        max_iter     : max EM iterations per initialization
        n_init       : number of random restarts (best silhouette kept)
        tol          : convergence threshold on centroid movement (radians)
        random_state : base seed (incremented per init run)

    Returns:
        best_labels    : (N,) cluster assignments from best run
        best_centroids : (k, d) final centroids from best run
        best_score     : silhouette score (cosine metric) of best run
    """
    # Safety: enforce unit norm
    norms = np.linalg.norm(X, axis=1, keepdims=True)
    X = X / np.maximum(norms, 1e-12)

    best_labels, best_centroids, best_score = None, None, -1.0

    for run in range(n_init):
        rng        = np.random.default_rng(random_state + run)
        centroids  = geodesic_kmeans_pp_init(X, k, rng)
        labels     = np.full(len(X), -1, dtype=int)

        for iteration in range(max_iter):
            # ── E-Step: Assign each point to nearest centroid by cosine sim
            # (equivalent to geodesic distance for unit vectors)
            similarities = X @ centroids.T          # (N, k) — fast matmul
            new_labels   = np.argmax(similarities, axis=1)

            # ── Convergence check
            if np.array_equal(new_labels, labels):
                break
            labels = new_labels

            # ── M-Step: Update centroids with curvature-aware Fréchet mean
            new_centroids = np.empty_like(centroids)
            for c_idx in range(k):
                mask = labels == c_idx
                if mask.sum() == 0:
                    # Empty cluster: reinitialize to random point
                    new_centroids[c_idx] = X[rng.integers(len(X))]
                else:
                    new_centroids[c_idx] = curvature_aware_centroid(
                        X[mask], centroids[c_idx], sigma=sigma
                    )

            # ── Centroid movement check (in geodesic radians)
            movements = geodesic_distances_to_centroid(
                new_centroids,
                centroids[0]  # placeholder — we check per centroid below
            )
            max_movement = max(
                geodesic_distances_to_centroid(
                    new_centroids[c:c+1], centroids[c]
                )[0]
                for c in range(k)
            )
            centroids = new_centroids
            if max_movement < tol:
                break

        # ── Score this run
        n_unique = len(np.unique(labels))
        if n_unique == k:
            score = silhouette_score(X, labels, metric='cosine')
            if score > best_score:
                best_score     = score
                best_labels    = labels.copy()
                best_centroids = centroids.copy()
        elif n_unique < k and best_labels is None:
            # Degenerate but record to avoid returning None
            best_labels    = labels.copy()
            best_centroids = centroids.copy()
            best_score     = -1.0

    cluster_sizes = {i: int((best_labels == i).sum()) for i in range(k)}
    print(f"✅ Spherical K-Means++ | k={k} | σ={sigma:.3f} | "
          f"Silhouette={best_score:.4f} | Cluster sizes: {cluster_sizes}")
    return best_labels, best_centroids, best_score


# ─────────────────────────────────────────────────────────────────────────────
# HYPERPARAMETER TUNING
# ─────────────────────────────────────────────────────────────────────────────

def tune_sigma(
    X: np.ndarray,
    k: int = 3,
    sigma_range: tuple = (0.05, 2.0),
    n_steps: int = 25,
    n_init: int = 10,
    random_state: int = 42
) -> tuple[float, list, list]:
    """
    Grid search over sigma values, maximizing silhouette score.
    Uses fewer n_init per step for speed; final run uses full n_init.

    Returns:
        best_sigma : float
        sigmas     : list of sigma values tested
        scores     : corresponding silhouette scores
    """
    sigmas = np.linspace(sigma_range[0], sigma_range[1], n_steps)
    scores = []

    print(f"── Tuning σ over {n_steps} values in [{sigma_range[0]}, {sigma_range[1]}] ──")
    for sigma in sigmas:
        _, _, score = run_spherical_kmeans(
            X, k=k, sigma=sigma, n_init=n_init, random_state=random_state
        )
        scores.append(score)

    best_sigma = float(sigmas[np.argmax(scores)])
    print(f"✅ Best σ: {best_sigma:.4f}  (Silhouette: {max(scores):.4f})\n")
    return best_sigma, sigmas.tolist(), scores


def tune_k(
    X: np.ndarray,
    k_range: range = range(2, 8),
    sigma: float = 0.5,
    n_init: int = 10,
    random_state: int = 42
) -> tuple[int, list, list]:
    """
    Sweep over k values to find optimal number of clusters.
    Uses silhouette score (higher = better separation).
    Also collect inertia (lower = tighter clusters) for elbow analysis.

    Note: For sentiment analysis with positive/negative/neutral labels,
    k=3 is ground truth. This sweep validates that assumption.
    """
    silhouette_scores = []
    inertias = []

    print(f"── Tuning k over {list(k_range)} ──")
    for k in k_range:
        _, centroids, score = run_spherical_kmeans(
            X, k=k, sigma=sigma, n_init=n_init, random_state=random_state
        )
        silhouette_scores.append(score)

        # Compute inertia: sum of squared geodesic distances to nearest centroid
        dist_matrix = np.column_stack([
            geodesic_distances_to_centroid(X, c) for c in centroids
        ])
        min_dists = dist_matrix.min(axis=1)
        inertias.append(float((min_dists ** 2).sum()))

    best_k = int(k_range[np.argmax(silhouette_scores)])
    print(f"✅ Best k: {best_k}  (Silhouette: {max(silhouette_scores):.4f})\n")
    return best_k, list(k_range), silhouette_scores, inertias

In [10]:
"""
evaluate.py
-----------
Full evaluation suite for clustering results.

Metrics:
    Silhouette Score     : cluster cohesion vs separation (−1 to 1, higher=better)
    Davies-Bouldin Index : intra-cluster / inter-cluster distance ratio (lower=better)
    Calinski-Harabasz    : between-cluster / within-cluster dispersion (higher=better)
    Adjusted Rand Index  : similarity to ground truth, corrected for chance (0 to 1)
    Normalized Mutual Info: information-theoretic agreement with ground truth (0 to 1)
    V-Measure            : harmonic mean of homogeneity and completeness

All plots saved to outputs/.
"""

import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from sklearn.metrics import (
    silhouette_score,
    davies_bouldin_score,
    calinski_harabasz_score,
    adjusted_rand_score,
    normalized_mutual_info_score,
    v_measure_score,
    confusion_matrix
)
import os

os.makedirs("outputs", exist_ok=True)

SENTIMENT_COLORS = {
    'positive': '#2196F3',
    'negative': '#F44336',
    'neutral':  '#4CAF50',
}
CLUSTER_COLORS = ['#1565C0', '#B71C1C', '#1B5E20', '#F57F17', '#4A148C']


# ─────────────────────────────────────────────────────────────────────────────
# METRIC COMPUTATION
# ─────────────────────────────────────────────────────────────────────────────

def evaluate_clustering(
    X: np.ndarray,
    labels: np.ndarray,
    true_labels_str: list[str],
    label_encoder: dict
) -> dict:
    """
    Compute full suite of internal and external clustering metrics.

    Internal metrics (no ground truth needed):
        - Silhouette Score (cosine metric)
        - Davies-Bouldin Index
        - Calinski-Harabasz Score

    External metrics (require ground truth):
        - Adjusted Rand Index
        - Normalized Mutual Information
        - V-Measure (homogeneity + completeness)
    """
    y_true = np.array([label_encoder[l] for l in true_labels_str])

    results = {
        # Internal
        'silhouette':        silhouette_score(X, labels, metric='cosine'),
        'davies_bouldin':    davies_bouldin_score(X, labels),
        'calinski_harabasz': calinski_harabasz_score(X, labels),
        # External
        'ari':        adjusted_rand_score(y_true, labels),
        'nmi':        normalized_mutual_info_score(y_true, labels),
        'v_measure':  v_measure_score(y_true, labels),
    }

    print("\n" + "─"*55)
    print("  CLUSTERING EVALUATION METRICS")
    print("─"*55)
    print("  Internal Metrics (no ground truth required):")
    print(f"    Silhouette Score     : {results['silhouette']:.4f}  "
          f"{'✅' if results['silhouette'] >= 0.7 else '⚠️ '} (target: ≥ 0.70)")
    print(f"    Davies-Bouldin Index : {results['davies_bouldin']:.4f}  "
          f"(lower is better)")
    print(f"    Calinski-Harabasz   : {results['calinski_harabasz']:.2f}  "
          f"(higher is better)")
    print("  External Metrics (vs ground truth labels):")
    print(f"    Adjusted Rand Index  : {results['ari']:.4f}  "
          f"(1.0 = perfect match)")
    print(f"    Normalized Mutual Info: {results['nmi']:.4f}")
    print(f"    V-Measure            : {results['v_measure']:.4f}")
    print("─"*55 + "\n")

    return results


# ─────────────────────────────────────────────────────────────────────────────
# VISUALIZATIONS
# ─────────────────────────────────────────────────────────────────────────────

def plot_umap_clusters(
    X_umap_2d: np.ndarray,
    pred_labels: np.ndarray,
    true_labels_str: list[str],
    cluster_to_sentiment: dict,
    save_path: str = "outputs/clusters_spherical.png"
):
    """
    Side-by-side 2D UMAP scatter plots:
      Left  — predicted cluster assignments (color by cluster)
      Right — ground truth labels (color by sentiment)
    """
    fig, axes = plt.subplots(1, 2, figsize=(15, 6))
    fig.suptitle("Spherical K-Means++ with Curvature-Aware Centroids\n"
                 "UMAP 2D Projection", fontsize=14, fontweight='bold', y=1.01)

    k = len(np.unique(pred_labels))

    # ── Left: Predicted clusters
    for c_id in range(k):
        mask = pred_labels == c_id
        name = cluster_to_sentiment.get(c_id, f"Cluster {c_id}")
        axes[0].scatter(
            X_umap_2d[mask, 0], X_umap_2d[mask, 1],
            c=CLUSTER_COLORS[c_id], label=f"Cluster {c_id} ({name})",
            s=60, edgecolors='k', linewidths=0.5, alpha=0.85
        )
    axes[0].set_title("Predicted Clusters", fontsize=12)
    axes[0].set_xlabel("UMAP Dimension 1")
    axes[0].set_ylabel("UMAP Dimension 2")
    axes[0].legend(fontsize=9)
    axes[0].grid(alpha=0.2)

    # ── Right: Ground truth
    for sentiment, color in SENTIMENT_COLORS.items():
        mask = np.array([t == sentiment for t in true_labels_str])
        marker = {'positive': 'o', 'negative': 's', 'neutral': '^'}[sentiment]
        axes[1].scatter(
            X_umap_2d[mask, 0], X_umap_2d[mask, 1],
            c=color, label=sentiment.capitalize(),
            marker=marker, s=60, edgecolors='k', linewidths=0.5, alpha=0.85
        )
    axes[1].set_title("Ground Truth Labels", fontsize=12)
    axes[1].set_xlabel("UMAP Dimension 1")
    axes[1].legend(fontsize=9)
    axes[1].grid(alpha=0.2)

    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"✅ Cluster scatter plot saved → {save_path}")


def plot_silhouette_per_sample(
    X: np.ndarray,
    labels: np.ndarray,
    save_path: str = "outputs/silhouette_per_sample.png"
):
    """
    Silhouette diagram — per-sample silhouette coefficients grouped by cluster.
    Wide bars = tight clusters. Uniform height = balanced cluster quality.
    """
    from sklearn.metrics import silhouette_samples

    sample_sils = silhouette_samples(X, labels, metric='cosine')
    mean_sil    = sample_sils.mean()
    k           = len(np.unique(labels))

    fig, ax = plt.subplots(figsize=(10, 6))
    y_lower = 10

    for c_id in range(k):
        c_silhouette = np.sort(sample_sils[labels == c_id])
        size_c = len(c_silhouette)
        y_upper = y_lower + size_c
        ax.fill_betweenx(
            np.arange(y_lower, y_upper),
            0, c_silhouette,
            facecolor=CLUSTER_COLORS[c_id],
            edgecolor=CLUSTER_COLORS[c_id],
            alpha=0.7,
            label=f"Cluster {c_id}"
        )
        ax.text(-0.05, y_lower + size_c / 2, str(c_id), fontsize=9)
        y_lower = y_upper + 10

    ax.axvline(x=mean_sil, color='crimson', linestyle='--', linewidth=1.5,
               label=f"Mean = {mean_sil:.4f}")
    ax.axvline(x=0.7, color='orange', linestyle=':', linewidth=1.5,
               label="Target = 0.70")
    ax.set_xlabel("Silhouette Coefficient")
    ax.set_ylabel("Cluster")
    ax.set_title("Per-Sample Silhouette Coefficients by Cluster", fontsize=12)
    ax.legend(loc='upper right', fontsize=9)
    ax.set_yticks([])
    ax.grid(alpha=0.2)
    plt.tight_layout()
    plt.savefig(save_path, dpi=150)
    plt.close()
    print(f"✅ Silhouette diagram saved → {save_path}")


def plot_sigma_sweep(
    sigmas: list,
    scores: list,
    best_sigma: float,
    save_path: str = "outputs/sigma_sweep.png"
):
    """Silhouette score vs. sigma — visualizes the curvature kernel sensitivity."""
    fig, ax = plt.subplots(figsize=(9, 4))
    ax.plot(sigmas, scores, marker='o', color='seagreen',
            linewidth=2, markersize=5)
    ax.axvline(best_sigma, color='crimson', linestyle='--', linewidth=1.5,
               label=f"Best σ = {best_sigma:.4f}")
    ax.set_xlabel("σ — Geodesic Gaussian Kernel Width (radians)", fontsize=11)
    ax.set_ylabel("Silhouette Score", fontsize=11)
    ax.set_title("Curvature-Aware Centroid: σ Sensitivity Analysis", fontsize=12)
    ax.legend(fontsize=10)
    ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(save_path, dpi=150)
    plt.close()
    print(f"✅ Sigma sweep plot saved → {save_path}")


def plot_k_sweep(
    k_range: list,
    silhouette_scores: list,
    inertias: list,
    best_k: int,
    save_path: str = "outputs/k_sweep.png"
):
    """Elbow + silhouette plot for k selection."""
    fig, axes = plt.subplots(1, 2, figsize=(13, 4))

    # Silhouette
    axes[0].plot(k_range, silhouette_scores, marker='o', color='steelblue',
                 linewidth=2)
    axes[0].axvline(best_k, color='crimson', linestyle='--',
                    label=f"Best k={best_k}")
    axes[0].axhline(0.7, color='orange', linestyle=':', label="Target 0.70")
    axes[0].set_xlabel("k (Number of Clusters)")
    axes[0].set_ylabel("Silhouette Score")
    axes[0].set_title("k vs. Silhouette Score")
    axes[0].legend()
    axes[0].grid(alpha=0.3)

    # Elbow (geodesic inertia)
    axes[1].plot(k_range, inertias, marker='o', color='darkorange', linewidth=2)
    axes[1].axvline(best_k, color='crimson', linestyle='--',
                    label=f"Best k={best_k}")
    axes[1].set_xlabel("k (Number of Clusters)")
    axes[1].set_ylabel("Geodesic Inertia (sum of squared arc distances)")
    axes[1].set_title("Elbow Method — Geodesic Inertia")
    axes[1].legend()
    axes[1].grid(alpha=0.3)

    plt.tight_layout()
    plt.savefig(save_path, dpi=150)
    plt.close()
    print(f"✅ k sweep plot saved → {save_path}")


def plot_comparison(
    old_scores: dict,
    new_score: float,
    save_path: str = "outputs/comparison.png"
):
    """Bar chart comparing all methods including the new pipeline."""
    all_scores = {**old_scores, "Spherical K-Means++\n+ Curvature-Aware\n(SBERT + UMAP)": new_score}
    methods    = list(all_scores.keys())
    values     = list(all_scores.values())

    colors = ['#90A4AE'] * (len(methods) - 1) + ['#1B5E20']

    fig, ax = plt.subplots(figsize=(10, 5))
    bars = ax.bar(methods, values, color=colors, edgecolor='black', width=0.4, zorder=3)

    for bar, val in zip(bars, values):
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 0.015,
            f"{val:.4f}",
            ha='center', va='bottom', fontsize=11, fontweight='bold'
        )

    ax.axhline(0.7, color='crimson', linestyle='--', linewidth=1.5,
               label='Target ≥ 0.70', zorder=4)
    ax.set_ylim(0, 1.05)
    ax.set_ylabel("Silhouette Score  (higher = better)", fontsize=11)
    ax.set_title("Silhouette Score Comparison — All Methods", fontsize=13, fontweight='bold')
    ax.legend(fontsize=10)
    ax.grid(axis='y', alpha=0.3, zorder=0)
    plt.tight_layout()
    plt.savefig(save_path, dpi=150)
    plt.close()
    print(f"✅ Comparison chart saved → {save_path}")


def infer_cluster_to_sentiment(labels: np.ndarray, true_labels_str: list[str]) -> dict:
    """
    Map cluster IDs → sentiment labels by majority vote.
    Needed because K-Means cluster indices are arbitrary.
    """
    k          = len(np.unique(labels))
    sentiments = sorted(set(true_labels_str))
    mapping    = {}

    for c_id in range(k):
        mask      = labels == c_id
        if mask.sum() == 0:
            mapping[c_id] = 'unknown'
            continue
        assigned  = [true_labels_str[i] for i in range(len(labels)) if mask[i]]
        majority  = max(set(assigned), key=assigned.count)
        mapping[c_id] = majority

    return mapping

In [12]:
"""
main.py
-------
Master orchestration script for the complete sentiment clustering pipeline.

FULL PIPELINE:
    1. Data Generation       → data/reviews.csv (600 reviews, 200/class)
    2. Preprocessing         → minimal cleaning (SBERT-safe)
    3. SBERT Embedding       → 384D semantic unit vectors
    4. UMAP Reduction        → 15D manifold (cosine metric)
    5. k Sweep               → validate k=3 with silhouette + geodesic inertia
    6. σ Sweep               → tune curvature kernel width
    7. Spherical K-Means++   → geodesic init + curvature-aware centroids
    8. Evaluation            → 6 metrics + full visualization suite

USAGE:
    python main.py [--generate] [--n_per_class 200] [--k 3] [--n_init 20]

FLAGS:
    --generate          : regenerate reviews.csv (skip if data already exists)
    --n_per_class N     : reviews per sentiment class (default: 200, total: 600)
    --k K               : number of clusters (default: 3; use 0 to auto-tune)
    --n_init N          : K-Means random restarts (default: 20)
    --umap_components N : UMAP output dimensions for clustering (default: 15)
    --data PATH         : path to CSV file (default: data/reviews.csv)
"""

import argparse
import os
import json
import numpy as np

from data_generator import generate_dataset
from preprocess     import load_and_preprocess
from vectorizer     import get_sbert_embeddings, get_umap_embedding, get_umap_2d
from geometric_kmeans import (
    run_spherical_kmeans,
    tune_sigma,
    tune_k
)
from evaluate import (
    evaluate_clustering,
    plot_umap_clusters,
    plot_silhouette_per_sample,
    plot_sigma_sweep,
    plot_k_sweep,
    plot_comparison,
    infer_cluster_to_sentiment
)

os.makedirs("outputs", exist_ok=True)
os.makedirs("data", exist_ok=True)


# ─────────────────────────────────────────────────────────────────────────────
# LABEL ENCODER (ground truth → int for ARI/NMI)
# ─────────────────────────────────────────────────────────────────────────────
LABEL_ENCODER = {'positive': 0, 'negative': 1, 'neutral': 2}


def print_banner(text: str):
    w = 65
    print("\n" + "═" * w)
    print(f"  {text}")
    print("═" * w)


def main(args):
    print_banner("SPHERICAL K-MEANS++ + SBERT + UMAP SENTIMENT PIPELINE")

    # ──────────────────────────────────────────────────────────────────────
    # STEP 1: DATA
    # ──────────────────────────────────────────────────────────────────────
    print_banner("Step 1 — Data")
    if args.generate or not os.path.exists(args.data):
        print(f"Generating {args.n_per_class * 3} reviews ({args.n_per_class}/class)...")
        generate_dataset(n_per_class=args.n_per_class, output_path=args.data)
    else:
        print(f"Using existing data: {args.data}")

    df           = load_and_preprocess(args.data)
    reviews      = df['clean_review'].tolist()
    true_labels  = df['label'].tolist()

    # ──────────────────────────────────────────────────────────────────────
    # STEP 2: SBERT EMBEDDINGS
    # ──────────────────────────────────────────────────────────────────────
    print_banner("Step 2 — SBERT Sentence Embeddings")
    X_sbert = get_sbert_embeddings(
    reviews,
    model_name=args.model_name,   
    batch_size=64,
    normalize_output=True
    )
    # Cache embeddings — SBERT inference is the bottleneck on CPU
    np.save("outputs/sbert_embeddings.npy", X_sbert)
    print(f"   Embeddings cached → outputs/sbert_embeddings.npy")

    # ──────────────────────────────────────────────────────────────────────
    # STEP 3: UMAP REDUCTION
    # ──────────────────────────────────────────────────────────────────────
    print_banner(f"Step 3 — UMAP Manifold Reduction (384D → {args.umap_components}D)")
    n_neighbors = max(5, min(50, int(np.sqrt(len(reviews)))))  # heuristic
    print(f"   Auto-selected n_neighbors={n_neighbors} (√N = {np.sqrt(len(reviews)):.1f})")

    X_umap = get_umap_embedding(
        X_sbert,
        n_components=args.umap_components,
        n_neighbors=n_neighbors,
        min_dist=0.05,
        random_state=42
    )

    # ──────────────────────────────────────────────────────────────────────
    # STEP 4: k SELECTION
    # ──────────────────────────────────────────────────────────────────────
    print_banner("Step 4 — Optimal k Selection (Silhouette + Geodesic Elbow)")
    if args.k == 0:
        best_k, k_vals, sil_scores, inertias = tune_k(
            X_umap, k_range=range(2, 8), sigma=0.5, n_init=10
        )
    else:
        best_k = args.k
        # Still run sweep for visualization
        _, k_vals, sil_scores, inertias = tune_k(
            X_umap, k_range=range(2, 8), sigma=0.5, n_init=10
        )
        print(f"   k fixed to {best_k} by user argument.")

    plot_k_sweep(k_vals, sil_scores, inertias, best_k,
                 save_path="outputs/k_sweep.png")

    # ──────────────────────────────────────────────────────────────────────
    # STEP 5: σ TUNING
    # ──────────────────────────────────────────────────────────────────────
    print_banner("Step 5 — Curvature Kernel σ Tuning")
    best_sigma, sigmas, sigma_scores = tune_sigma(
        X_umap, k=best_k, sigma_range=(0.05, 2.0), n_steps=30,
        n_init=10, random_state=42
    )
    plot_sigma_sweep(sigmas, sigma_scores, best_sigma,
                     save_path="outputs/sigma_sweep.png")

    # ──────────────────────────────────────────────────────────────────────
    # STEP 6: FINAL SPHERICAL K-MEANS++
    # ──────────────────────────────────────────────────────────────────────
    print_banner(f"Step 6 — Final Spherical K-Means++ (k={best_k}, σ={best_sigma:.4f})")
    final_labels, final_centroids, final_score = run_spherical_kmeans(
        X_umap,
        k=best_k,
        sigma=best_sigma,
        max_iter=500,
        n_init=args.n_init,
        random_state=42
    )

    # ──────────────────────────────────────────────────────────────────────
    # STEP 7: EVALUATION
    # ──────────────────────────────────────────────────────────────────────
    print_banner("Step 7 — Evaluation")
    metrics = evaluate_clustering(X_umap, final_labels, true_labels, LABEL_ENCODER)

    # Save metrics as JSON
    with open("outputs/metrics.json", "w") as f:
        json.dump({k: round(float(v), 6) for k, v in metrics.items()}, f, indent=2)
    print("   Metrics saved → outputs/metrics.json")

    # ──────────────────────────────────────────────────────────────────────
    # STEP 8: VISUALIZATIONS
    # ──────────────────────────────────────────────────────────────────────
    print_banner("Step 8 — Visualizations")

    # 2D UMAP for plotting (separate from clustering UMAP)
    print("   Generating 2D UMAP projection for visualization...")
    X_umap_2d = get_umap_2d(X_sbert, n_neighbors=n_neighbors, min_dist=0.1)

    cluster_map = infer_cluster_to_sentiment(final_labels, true_labels)
    print(f"   Cluster → Sentiment mapping: {cluster_map}")

    plot_umap_clusters(
        X_umap_2d, final_labels, true_labels, cluster_map,
        save_path="outputs/clusters_spherical.png"
    )
    plot_silhouette_per_sample(
        X_umap, final_labels,
        save_path="outputs/silhouette_per_sample.png"
    )
    plot_comparison(
        old_scores={
            "Standard\nK-Means\n(TF-IDF)":  0.0104,
            "Geometric\nK-Means\n(TF-IDF)": 0.0104,
        },
        new_score=final_score,
        save_path="outputs/comparison.png"
    )

    # ──────────────────────────────────────────────────────────────────────
    # FINAL SUMMARY
    # ──────────────────────────────────────────────────────────────────────
    print_banner("FINAL RESULTS")
    print(f"  Dataset size           : {len(reviews)} reviews ({len(reviews)//3}/class)")
    print(f"  Embedding              : {args.model_name}")
    print(f"  UMAP dimensions        : {args.umap_components}D (n_neighbors={n_neighbors})")
    print(f"  Optimal k              : {best_k}")
    print(f"  Optimal σ              : {best_sigma:.4f}")
    print(f"  n_init                 : {args.n_init}")
    print(f"")
    print(f"  Silhouette Score       : {metrics['silhouette']:.4f}  "
          f"{'✅ TARGET MET' if metrics['silhouette'] >= 0.7 else '❌ Below target'}")
    print(f"  Davies-Bouldin Index   : {metrics['davies_bouldin']:.4f}  (lower = better)")
    print(f"  Calinski-Harabasz      : {metrics['calinski_harabasz']:.2f}  (higher = better)")
    print(f"  Adjusted Rand Index    : {metrics['ari']:.4f}")
    print(f"  Norm. Mutual Info      : {metrics['nmi']:.4f}")
    print(f"  V-Measure              : {metrics['v_measure']:.4f}")
    print(f"")
    baseline = 0.0104
    print(f"  Improvement over baseline : "
          f"+{((metrics['silhouette'] - baseline)/baseline*100):.0f}%  "
          f"({baseline:.4f} → {metrics['silhouette']:.4f})")
    print("═" * 65 + "\n")

    print("  Output files:")
    output_files = [
        "outputs/clusters_spherical.png   → UMAP 2D scatter (predicted + ground truth)",
        "outputs/silhouette_per_sample.png → Per-sample silhouette diagram",
        "outputs/sigma_sweep.png           → Curvature kernel σ sensitivity",
        "outputs/k_sweep.png               → k selection (silhouette + elbow)",
        "outputs/comparison.png            → Method comparison bar chart",
        "outputs/sbert_embeddings.npy      → Cached SBERT embeddings (384D)",
        "outputs/metrics.json              → All metrics as JSON",
    ]
    for f in output_files:
        print(f"    {f}")
    print()


if __name__ == "__main__":
    parser = argparse.ArgumentParser(description="Spherical K-Means++ Sentiment Pipeline")
    parser.add_argument('--generate',       action='store_true',
                        help='Regenerate reviews.csv from templates')
    parser.add_argument('--n_per_class',    type=int, default=200,
                        help='Reviews per sentiment class (total = 3x this)')
    parser.add_argument('--k',              type=int, default=3,
                        help='Number of clusters (0 = auto-tune)')
    parser.add_argument('--n_init',         type=int, default=20,
                        help='K-Means random restarts')
    parser.add_argument('--umap_components',type=int, default=15,
                        help='UMAP output dimensions for clustering (not visualization)')
    parser.add_argument('--data',           type=str, default='data/reviews.csv',
                        help='Path to reviews CSV (review, label columns)')
    parser.add_argument('--model_name',     type=str, default='siebert/sentiment-roberta-large-english',
                    help='SBERT model name from HuggingFace')
    args, unknown = parser.parse_known_args()
    main(args)


═════════════════════════════════════════════════════════════════
  SPHERICAL K-MEANS++ + SBERT + UMAP SENTIMENT PIPELINE
═════════════════════════════════════════════════════════════════

═════════════════════════════════════════════════════════════════
  Step 1 — Data
═════════════════════════════════════════════════════════════════
Using existing data: data/reviews.csv
✅ Loaded 600 reviews | Distribution: {'positive': 200, 'negative': 200, 'neutral': 200}

═════════════════════════════════════════════════════════════════
  Step 2 — SBERT Sentence Embeddings
═════════════════════════════════════════════════════════════════
── Loading SBERT model: siebert/sentiment-roberta-large-english ──


No sentence-transformers model found with name siebert/sentiment-roberta-large-english. Creating a new one with mean pooling.


Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: siebert/sentiment-roberta-large-english
Key                             | Status     | 
--------------------------------+------------+-
classifier.out_proj.weight      | UNEXPECTED | 
classifier.dense.weight         | UNEXPECTED | 
classifier.out_proj.bias        | UNEXPECTED | 
classifier.dense.bias           | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
pooler.dense.bias               | MISSING    | 
pooler.dense.weight             | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


── Encoding 600 sentences (batch_size=64) ──


...

✅ SBERT embeddings: (600, 1024)  |  dtype: float32
   Embeddings cached → outputs/sbert_embeddings.npy

═════════════════════════════════════════════════════════════════
  Step 3 — UMAP Manifold Reduction (384D → 15D)
═════════════════════════════════════════════════════════════════
   Auto-selected n_neighbors=24 (√N = 24.5)
── UMAP reduction: 384D → 15D  (n_neighbors=24, min_dist=0.05) ──


C:\Users\Adwaith\AppData\Roaming\Python\Python313\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


✅ UMAP embedding: (600, 15)  |  L2-normalized ✓

═════════════════════════════════════════════════════════════════
  Step 4 — Optimal k Selection (Silhouette + Geodesic Elbow)
═════════════════════════════════════════════════════════════════
── Tuning k over [2, 3, 4, 5, 6, 7] ──
✅ Spherical K-Means++ | k=2 | σ=0.500 | Silhouette=0.7568 | Cluster sizes: {0: 292, 1: 308}
✅ Spherical K-Means++ | k=3 | σ=0.500 | Silhouette=0.7332 | Cluster sizes: {0: 51, 1: 257, 2: 292}
✅ Spherical K-Means++ | k=4 | σ=0.500 | Silhouette=0.7594 | Cluster sizes: {0: 140, 1: 292, 2: 51, 3: 117}
✅ Spherical K-Means++ | k=5 | σ=0.500 | Silhouette=0.8091 | Cluster sizes: {0: 218, 1: 81, 2: 51, 3: 211, 4: 39}
✅ Spherical K-Means++ | k=6 | σ=0.500 | Silhouette=0.8910 | Cluster sizes: {0: 51, 1: 78, 2: 211, 3: 39, 4: 140, 5: 81}
✅ Spherical K-Means++ | k=7 | σ=0.500 | Silhouette=0.8737 | Cluster sizes: {0: 51, 1: 78, 2: 211, 3: 39, 4: 79, 5: 81, 6: 61}
✅ Best k: 6  (Silhouette: 0.8910)

   k fixed to 3 by user arg

C:\Users\Adwaith\AppData\Roaming\Python\Python313\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


   Cluster → Sentiment mapping: {0: 'positive', 1: 'negative', 2: 'neutral'}
✅ Cluster scatter plot saved → outputs/clusters_spherical.png
✅ Silhouette diagram saved → outputs/silhouette_per_sample.png
✅ Comparison chart saved → outputs/comparison.png

═════════════════════════════════════════════════════════════════
  FINAL RESULTS
═════════════════════════════════════════════════════════════════
  Dataset size           : 600 reviews (200/class)
  Embedding              : siebert/sentiment-roberta-large-english
  UMAP dimensions        : 15D (n_neighbors=24)
  Optimal k              : 3
  Optimal σ              : 0.0500
  n_init                 : 20

  Silhouette Score       : 0.7332  ✅ TARGET MET
  Davies-Bouldin Index   : 0.5589  (lower = better)
  Calinski-Harabasz      : 875.02  (higher = better)
  Adjusted Rand Index    : 0.5008
  Norm. Mutual Info      : 0.5626
  V-Measure              : 0.5626

  Improvement over baseline : +6950%  (0.0104 → 0.7332)
═══════════════════════════